# 02 · LoRA fine-tune — Qwen3-ASR-1.7B on Vietnamese

Fine-tunes `Qwen/Qwen3-ASR-1.7B-hf` with a **PEFT LoRA** adapter on a ~34 h
**multi-source Vietnamese mixture** (viVoice + VIVOS + Common Voice 17 `vi` +
VLSP 2020), validates on the mixture's `val`, and reports **before/after WER**
on the viVoice `test` split used by `01_eval_baseline.ipynb` (section 7) plus
the external benchmark suite (section 9).

> Sections 7 and 9 measure different things: viVoice `test` is speaker-disjoint
> and **unseen**, while the section 9 benchmarks are **in-domain** after this
> fine-tune. See `mixture.py` for how the eval rows are kept clean.

**16 GB VRAM budget:** batch size 1 · gradient accumulation 16 · gradient
checkpointing · bf16 · `adamw_torch_fused`. LoRA is attached to the **language
model decoder** — attention *and* MLP (audio encoder stays frozen).

**Baked-in facts from setup:**
- Audio passed to the processor as a numpy array (FFmpeg 4 file-decode is buggy here).
- Audio feature keys: `input_features`, `input_features_mask`.
- Requires `transformers>=5.14`.

## What changed since run 1

Run 1 was a null result: held-out viVoice WER 5.958% → 5.938%, a difference of
2 word errors in 10,155 (bootstrap 95% CI [-0.42%, +0.37%]). It also *lost*
4.5 points on VIVOS, 7.19% → 11.64%.

Root cause was in the data, not the optimizer — training was healthy throughout
(val loss 0.363 → 0.341 → 0.334, monotone). VIVOS ships ALL-CAPS transcripts and
was 59% of the mixture, so the model learned to condition casing on acoustic
domain and emitted uppercase for all 760 VIVOS test clips. Uppercase Vietnamese
has no whole-word tokens in the Qwen vocabulary, so generation dropped to roughly
character granularity in a token space with almost no language-model prior —
turning a cosmetic mismatch into phonetic errors ("trót" → "TÓT"). Fixed in
`mixture.py::normalize_train_text`; the cache is version-stamped so `data/vi_mix/`
rebuilds rather than being reused.

Three changes for run 2:
1. **Transcript casing normalized on ingest** — the actual bug.
2. **LoRA extended to the decoder MLP** — 6.4M → 17.4M trainable params (0.31% → 0.85%).
3. **LR 2e-4 → 1e-4** to match the larger adapted surface.

Only change 1 addresses a known defect; 2 and 3 are a bet that the null result
on viVoice was partly capacity. If run 2 fixes VIVOS but viVoice `test` still
does not move, the honest read is that 34 h is not enough to improve a model
already at 5.96% WER — not that the recipe needs more tuning.

Run 1's numbers live in `results/`; rerunning overwrites them, so copy that
directory aside first if you want the before/after side by side.

Run top-to-bottom (Kernel → Restart & Run All).

In [2]:
import os, random, numpy as np, torch
from dotenv import load_dotenv
load_dotenv()
assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in .env (see .env.example)"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

torch 2.8.0+cu128 | device: cuda | NVIDIA GeForce RTX 5080


## 1 · Build / load the training mixture
Reuses the shared `data/vi_asr/` viVoice cache and builds `data/vi_mix/`
(viVoice + VIVOS + Common Voice + VLSP 2020). Cache-aware: the first run
streams ~5 GB and writes 16 kHz wavs; later runs are a no-op.

In [3]:
import data_prep, mixture

# viVoice cache (also feeds the mixture) — no-op if already built
data_prep.prepare_dataset(target_hours=8.0)

# Multi-source training mixture: viVoice + VIVOS + Common Voice 17 vi + VLSP 2020.
# Cache-aware; the first build streams ~5 GB and writes 16 kHz wavs (~1 h).
mixture.prepare_mixture()

mix = data_prep.load_splits("data/vi_mix")
train, val = mix["train"], mix["val"]

# Drop the handful of very long clips. data_prep caps viVoice segments, but the
# streamed sources are uncapped: VLSP ships four clips of 61-81 s, well past the
# ~57 s longest clip whose memory use has actually been measured (6.7 GB peak).
# They are 0.02% of the mixture and the only untested OOM path in a ~7 h run.
MAX_TRAIN_S = 60.0
_before = len(train)
train = train.filter(lambda r: r["duration"] <= MAX_TRAIN_S)
print(f"dropped {_before - len(train)} train clips > {MAX_TRAIN_S:.0f}s "
      f"({_before} -> {len(train)})")

# `test` stays the viVoice held-out split so the before/after against
# results/baseline_metrics.json remains apples-to-apples. It is the only test set
# here that is still speaker-disjoint AND unseen. The external benchmarks are
# evaluated separately in section 9.
test = data_prep.load_splits("data/vi_asr")["test"]

print({"train": len(train), "val": len(val), "test (viVoice)": len(test)})
print(mixture.summarize(list(train)).to_string(index=False))


[data_prep] cache hit at data/vi_asr; skipping build.
[mixture] cache at data/vi_mix is v0, need v2 — rebuilding (~1 h, re-streams the non-viVoice sources).
[mixture] vivoice: reused 1302 rows from data/vi_asr


vivos:train: 0clip [00:00, ?clip/s]

cmv_vi:train: 0clip [00:00, ?clip/s]

cmv_vi:validation: 0clip [00:00, ?clip/s]

vlsp2020_100h:train: 0clip [00:00, ?clip/s]

...

[mixture] test: 390 rows -> data/vi_mix/test


Saving the dataset (0/1 shards):   0%|          | 0/276 [00:00<?, ? examples/s]

[mixture] heldout vlsp2020_100h: 276 rows -> data/vi_mix/heldout_vlsp2020_100h

[mixture] training composition:
       source     n  hours
       cmv_vi  2298  2.895
      vivoice   980  6.409
        vivos 11660 14.921
vlsp2020_100h  4947  9.849


Filter:   0%|          | 0/19885 [00:00<?, ? examples/s]

dropped 4 train clips > 60s (19885 -> 19881)
{'train': 19881, 'val': 1014, 'test (viVoice)': 114}
       source     n  hours
       cmv_vi  2298  2.895
      vivoice   980  6.409
        vivos 11660 14.921
vlsp2020_100h  4943  9.775


## 2 · Load the base model + processor

In [4]:
from transformers import AutoProcessor, AutoModelForMultimodalLM
MODEL_ID = "Qwen/Qwen3-ASR-1.7B-hf"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, attn_implementation="sdpa", device_map=DEVICE,
)
print("loaded", type(model).__name__, model.dtype, model.device)

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

loaded Qwen3ASRForConditionalGeneration torch.bfloat16 cuda:0


## 3 · Attach the LoRA adapter (decoder attention + MLP)

`q/k/v_proj` exist in both the audio encoder and the text decoder, so we match
**only** the `model.language_model` projections with a regex — the audio encoder
stays frozen.

The MLP projections (`gate/up/down_proj`) are included alongside attention. Run 1
adapted attention only (6.4M params, 0.31%) and moved the held-out viVoice WER by
2 word errors in 10,155 — indistinguishable from noise. The decoder MLP is where a
transformer stores most of its lexical knowledge, which is what adapting to a new
language's vocabulary actually needs. Learning rate drops 2e-4 → 1e-4 to match the
larger adapted surface.

In [5]:
from peft import LoraConfig, get_peft_model

# Full-match regex over module names -> only the 28 decoder layers, never the
# audio encoder. 4 attention + 3 MLP projections per layer = 196 total.
TARGET_RE = (r"model\.language_model\.layers\.\d+\."
             r"(self_attn\.(q_proj|k_proj|v_proj|o_proj)"
             r"|mlp\.(gate_proj|up_proj|down_proj))")

model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()   # required for grad checkpointing + PEFT

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM", target_modules=TARGET_RE,
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

adapted = [n for n, _ in model.named_modules() if n.endswith("lora_A.default")]
assert len(adapted) == 28 * 7, f"expected 196 LoRA-adapted projections, got {len(adapted)}"
# The audio encoder has q/k/v_proj and fc1/fc2 of its own, so a regex that
# widened by accident would silently unfreeze it — and the encoder is exactly
# what this run means to hold fixed.
assert not [n for n in adapted if ".language_model." not in n], \
    "LoRA leaked outside model.language_model — audio encoder must stay frozen"
print("LoRA-adapted projections:", len(adapted))

trainable params: 17,432,576 || all params: 2,055,485,056 || trainable%: 0.8481
LoRA-adapted projections: 196


## 4 · Data collator (on-the-fly)

For each raw example we build the ASR request (audio + prompt), then append the
target transcription tokens + EOS as **labels**, masking the prompt with `-100`.
Features are computed on the fly (not serialized to disk) since the dataset is small.

In [6]:
EOS_ID = processor.tokenizer.eos_token_id
PAD_ID = processor.tokenizer.pad_token_id

def collate(features):
    ids_list, lbl_list, feats, feat_masks = [], [], [], []
    for ex in features:
        arr, _ = data_prep.read_audio(ex["audio_path"])   # 16 kHz mono
        req = processor.apply_transcription_request(audio=arr, language="Vietnamese")
        p_ids = req["input_ids"][0]
        tgt = processor.tokenizer(ex["text"], add_special_tokens=False,
                                  return_tensors="pt")["input_ids"][0]
        tgt = torch.cat([tgt, torch.tensor([EOS_ID], dtype=tgt.dtype)])
        ids = torch.cat([p_ids, tgt])
        lbl = torch.cat([torch.full((len(p_ids),), -100, dtype=torch.long), tgt])
        ids_list.append(ids); lbl_list.append(lbl)
        feats.append(req["input_features"][0])
        feat_masks.append(req["input_features_mask"][0])

    maxlen = max(len(x) for x in ids_list)
    def pad1(x, val):
        return torch.cat([x, torch.full((maxlen - len(x),), val, dtype=x.dtype)])
    input_ids = torch.stack([pad1(x, PAD_ID) for x in ids_list])
    labels = torch.stack([pad1(x, -100) for x in lbl_list])
    attn = torch.stack([
        torch.cat([torch.ones(len(x), dtype=torch.long),
                   torch.zeros(maxlen - len(x), dtype=torch.long)]) for x in ids_list])

    maxT = max(f.shape[-1] for f in feats)
    input_features = torch.stack(
        [torch.cat([f, torch.zeros(f.shape[0], maxT - f.shape[-1], dtype=f.dtype)], dim=-1)
         for f in feats])
    input_features_mask = torch.stack(
        [torch.cat([m, torch.zeros(maxT - len(m), dtype=m.dtype)]) for m in feat_masks])

    return {"input_ids": input_ids, "attention_mask": attn, "labels": labels,
            "input_features": input_features, "input_features_mask": input_features_mask}

# sanity check on 2 examples
_b = collate([train[0], train[1]])
print({k: tuple(v.shape) for k, v in _b.items()})

{'input_ids': (2, 245), 'attention_mask': (2, 245), 'labels': (2, 245), 'input_features': (2, 128, 1300), 'input_features_mask': (2, 1300)}


## 5 · Training configuration

In [7]:
from transformers import Trainer, TrainerCallback, TrainingArguments

# ~19.9k samples / 16 accumulation = ~1243 optimizer steps per epoch, so one epoch
# is ~1.2k weight updates, not 1. Evaluating every 400 steps samples the
# validation curve 3 times -- enough to tell "still improving" from "overfitting",
# and to give load_best_model_at_end real candidates to choose from.
EVAL_EVERY = 400
LOG_EVERY = 5


class LossPrinter(TrainerCallback):
    """Print train and validation loss on their own lines.

    Needed because transformers' notebook progress table only writes a
    training-loss row when ``eval_strategy == "no"`` (see on_log in
    transformers/utils/notebook.py). With eval enabled the table is written
    only at eval time, so without this the first loss shown is at step 400.
    """

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        step = f"{state.global_step:5d}/{state.max_steps}"
        if "loss" in logs:          # training log, every logging_steps
            print(f"[train] {step}  loss {logs['loss']:.4f}"
                  f"  lr {logs.get('learning_rate', float('nan')):.2e}", flush=True)
        if "eval_loss" in logs:     # evaluation log, every eval_steps
            print(f"[ val ] {step}  loss {logs['eval_loss']:.4f}"
                  f"  ({logs.get('eval_runtime', 0):.0f}s)", flush=True)


args = TrainingArguments(
    output_dir="checkpoints/vi_lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    # gradient_accumulation_steps=32,
    num_train_epochs=1,
    # 2e-4 was tuned for attention-only (6.4M params). Section 3 now adapts the
    # MLP too (17.4M, 2.7x the trainable surface), so halve it: the same step
    # size over more directions moves the decoder's lexical weights further
    # than intended, and those weights are what hold the language-model prior.
    learning_rate=1e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    bf16=True,
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    logging_steps=LOG_EVERY,
    eval_strategy="steps",
    eval_steps=EVAL_EVERY,
    save_strategy="steps",
    save_steps=EVAL_EVERY,          # must equal eval_steps for load_best_model_at_end
    # Make the val split actually do its job: keep the checkpoint that validated
    # best rather than whichever ran last.
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,             # best + latest; 1 can delete the best
    per_device_eval_batch_size=1,   # 16 GB: long clips OOM at the default 8
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,   # our collator consumes the raw columns
    dataloader_num_workers=2,
)
trainer = Trainer(model=model, args=args, train_dataset=train,
                  eval_dataset=val, data_collator=collate,
                  callbacks=[LossPrinter()])

# ceil, matching how Trainer counts: the trailing partial accumulation group
# still fires an optimizer step.
steps_per_epoch = -(-len(train) // args.gradient_accumulation_steps)
total_steps = int(steps_per_epoch * args.num_train_epochs)
print(f"steps/epoch {steps_per_epoch}  total updates {total_steps}  "
      f"train-loss lines {total_steps // LOG_EVERY}  evals {total_steps // EVAL_EVERY}")


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


steps/epoch 1243  total updates 1243  train-loss lines 248  evals 3


## 6 · Train
Watch VRAM stay < 16 GB. Measured peak on this box was 6.7 GB with clips up to
~57 s, and cell 3 now drops anything over 60 s, so there is real headroom.
If you do OOM, raise `gradient_accumulation_steps` or lower `MAX_TRAIN_S` in
cell 3 — no rebuild needed, it filters the cached mixture in place.

Validation runs every `EVAL_EVERY` (400) steps. Watch `eval_loss`: if it is
still falling at the end the run is undertrained and a third epoch is
justified; if it turns upward, `load_best_model_at_end` will hand back the
best checkpoint rather than the last one, so the final adapter may not be the
one trained longest.

In [8]:
trainer.train()
trainer.save_model("checkpoints/vi_lora")   # saves the LoRA adapter
print("peak VRAM GB:", round(torch.cuda.max_memory_allocated() / 1e9, 2))
print("adapter saved -> checkpoints/vi_lora")

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
400,0.172864,0.357190
800,0.222411,0.334994
1200,0.215664,0.329175
1243,0.235969,0.328932


[train]     5/1243  loss 2.0951  lr 1.05e-05
[train]    10/1243  loss 1.6015  lr 2.37e-05
[train]    15/1243  loss 1.0258  lr 3.68e-05
[train]    20/1243  loss 0.3852  lr 5.00e-05
[train]    25/1243  loss 0.3972  lr 6.32e-05
[train]    30/1243  loss 0.3031  lr 7.63e-05
[train]    35/1243  loss 0.2603  lr 8.95e-05
[train]    40/1243  loss 0.3495  lr 1.00e-04
[train]    45/1243  loss 0.2033  lr 1.00e-04
[train]    50/1243  loss 0.3411  lr 1.00e-04
[train]    55/1243  loss 0.2739  lr 1.00e-04
[train]    60/1243  loss 0.3363  lr 9.99e-05
[train]    65/1243  loss 0.2740  lr 9.99e-05
[train]    70/1243  loss 0.2392  lr 9.98e-05
[train]    75/1243  loss 0.3628  lr 9.98e-05
[train]    80/1243  loss 0.2587  lr 9.97e-05
[train]    85/1243  loss 0.3561  lr 9.96e-05
[train]    90/1243  loss 0.2400  lr 9.96e-05
[train]    95/1243  loss 0.2551  lr 9.95e-05
[train]   100/1243  loss 0.2916  lr 9.94e-05
[train]   105/1243  loss 0.1901  lr 9.93e-05
[train]   110/1243  loss 0.3080  lr 9.91e-05
[train]   

## 7 · Evaluate the fine-tuned model on the test split

In [9]:
from vi_norm import normalize_vi, wer_cer   # shared with 01_eval_baseline.ipynb

model.config.use_cache = True
model.eval()
processor.tokenizer.padding_side = "left"   # required for correct batched generation
BATCH_SIZE = 8   # lower to 4 if you hit CUDA OOM

@torch.no_grad()
def transcribe_batch(rows, batch_size=BATCH_SIZE, max_new_tokens=440):
    from tqdm.auto import tqdm
    rows = list(rows)
    order = sorted(range(len(rows)), key=lambda i: rows[i]["duration"])
    hyps = [None] * len(rows)
    for s in tqdm(range(0, len(order), batch_size), desc="eval (batched)"):
        chunk = order[s:s + batch_size]
        arrs = [data_prep.read_audio(rows[i]["audio_path"])[0] for i in chunk]
        inputs = processor.apply_transcription_request(
            audio=arrs, language="Vietnamese").to(model.device, model.dtype)
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        gen = out[:, inputs["input_ids"].shape[1]:]
        for i, txt in zip(chunk, processor.decode(gen, return_format="transcription_only")):
            hyps[i] = txt
    return hyps

In [10]:
import pathlib
import pandas as pd

pathlib.Path("results").mkdir(exist_ok=True)   # first write of the notebook

test_rows = list(test)
hyps = transcribe_batch(test_rows)   # batched, ~3x faster
rows = [{"audio_file": os.path.basename(r["audio_path"]), "audio_path": r["audio_path"],
         "channel": r["channel"], "bucket": r["bucket"], "duration": round(r["duration"], 2),
         "ref": r["text"], "hyp": h}
        for r, h in zip(test_rows, hyps)]
df = pd.DataFrame(rows)
df.to_csv("results/lora_predictions.csv", index=False)
lora_overall = wer_cer(df["ref"], df["hyp"])
lora_by = {b: wer_cer(g["ref"], g["hyp"]) for b, g in df.groupby("bucket")}
print("LoRA OVERALL:", lora_overall)
for b, m in lora_by.items():
    print(f"  {b}: WER={m['wer']:.3f} CER={m['cer']:.3f} (n={m['n']})")

eval (batched):   0%|          | 0/15 [00:00<?, ?it/s]

LoRA OVERALL: {'wer': 0.06026587887740029, 'cer': 0.02838768948093707, 'n': 114, 'wer_legacy': 0.06475690942102078, 'cer_legacy': 0.03039693428584946}
  30-60: WER=0.059 CER=0.028 (n=20)
  5-30: WER=0.061 CER=0.029 (n=94)


## 8 · Before / after comparison

In [11]:
import json, pathlib
pathlib.Path("results").mkdir(exist_ok=True)
with open("results/lora_metrics.json", "w", encoding="utf-8") as f:
    json.dump({"overall": lora_overall, "by_bucket": lora_by}, f, ensure_ascii=False, indent=2)

try:
    baseline = json.load(open("results/baseline_metrics.json"))
    comp = pd.DataFrame({
        "metric": ["WER", "CER"],
        "baseline": [baseline["overall"]["wer"], baseline["overall"]["cer"]],
        "lora": [lora_overall["wer"], lora_overall["cer"]],
    })
    comp["abs_delta"] = comp["lora"] - comp["baseline"]
    comp["rel_%"] = 100 * comp["abs_delta"] / comp["baseline"]
    print(comp.to_string(index=False))
except FileNotFoundError:
    print("Run 01_eval_baseline.ipynb first to get baseline_metrics.json for the comparison.")
    comp = None
comp

metric  baseline     lora  abs_delta     rel_%
   WER  0.059577 0.060266   0.000689  1.157025
   CER  0.030340 0.028388  -0.001952 -6.434519


,metric,baseline,lora,abs_delta,rel_%
0,WER,0.059577,0.060266,0.000689,1.157025
1,CER,0.030340,0.028388,-0.001952,-6.434519


## 9 · External benchmarks — before / after

Scores the fine-tuned adapter on the same benchmark suite as notebook 1 and
diffs it against `results/bench_metrics.json`.

> **These are no longer zero-shot.** The mixture trains on the *train* splits of
> VIVOS and Common Voice, and on 90% of the VLSP corpus (by transcript hash), so
> all three test sets are now **in-domain**. The improvement here measures
> adaptation to these datasets, not general Vietnamese ASR gains. The genuinely
> held-out, speaker-disjoint number is the viVoice `test` result in section 7.
>
> The eval rows themselves stay clean: VIVOS/Common Voice test splits are
> untouched, and the VLSP slice is hash-disjoint from everything trained on.

If the baseline file was produced before the VLSP held-out slice existed, the
cell below says so — re-run notebook 1 section 8 to refresh it.

In [12]:
import bench

processor.tokenizer.padding_side = "left"
lora_bench, lora_bench_frames = bench.run_benchmarks(
    model, processor, ["vivos", "cmv_vi", "vlsp2020_100h"],
    token=os.environ.get("HF_TOKEN"), batch_size=8,
)

pathlib.Path("results").mkdir(exist_ok=True)
for name, f in lora_bench_frames.items():
    f.to_csv(f"results/lora_bench_{name}_predictions.csv", index=False)
with open("results/lora_bench_metrics.json", "w", encoding="utf-8") as f:
    json.dump(lora_bench, f, ensure_ascii=False, indent=2)
print("saved -> results/lora_bench_metrics.json")



=== vivos · htdung167/vivos-preprocessed-v2 [test] ===
    Official VIVOS test split, 760 utterances.


loading vivos: 0it [00:00, ?it/s]

transcribing (batched):   0%|          | 0/95 [00:00<?, ?it/s]

   WER=0.0603  CER=0.0278  (legacy WER=0.0603)  n=760

=== cmv_vi · fixie-ai/common_voice_17_0 [test] ===
    Common Voice 17.0 Vietnamese test split, 1274 utterances.


loading cmv_vi: 0it [00:00, ?it/s]

transcribing (batched):   0%|          | 0/160 [00:00<?, ?it/s]

   WER=0.0987  CER=0.0449  (legacy WER=0.0990)  n=1274

=== vlsp2020_100h · data/vi_mix/heldout_vlsp2020_100h [test] ===
    VLSP 2020 VinBigData 100h corpus, 5% held-out slice assigned by transcript hash. Own benchmark, not PhoWhisper's T1/T2. ~96% transcript accuracy -> read the delta, not the absolute.


loading vlsp2020_100h: 0it [00:00, ?it/s]

transcribing (batched):   0%|          | 0/35 [00:00<?, ?it/s]

   WER=0.0957  CER=0.0518  (legacy WER=0.0960)  n=276
saved -> results/lora_bench_metrics.json


In [13]:
try:
    base_bench = json.load(open("results/bench_metrics.json"))
except FileNotFoundError:
    base_bench = {}
    print("No results/bench_metrics.json — run notebook 1 section 8 for the baseline.")

rows = []
for name, l in lora_bench.items():
    b = base_bench.get(name)
    if b and b.get("source") != l.get("source"):
        print(f"WARNING {name}: baseline came from {b.get('source')!r} but this run "
              f"used {l.get('source')!r}. Re-run notebook 1 section 8 — the "
              f"comparison below is not valid for this row.")
    rows.append({
        "dataset": name,
        "n": l["n"],
        "baseline WER %": round(100 * b["wer"], 2) if b else None,
        "LoRA WER %": round(100 * l["wer"], 2),
        "abs delta": round(100 * (l["wer"] - b["wer"]), 2) if b else None,
        "rel %": round(100 * (l["wer"] - b["wer"]) / b["wer"], 1) if b else None,
        "zero-shot?": "no — trained on this dataset",
    })
bench_cmp = pd.DataFrame(rows)
print()
print(bench_cmp.to_string(index=False))
bench_cmp



      dataset    n  baseline WER %  LoRA WER %  abs delta  rel %                   zero-shot?
        vivos  760            7.19        6.03      -1.15  -16.0 no — trained on this dataset
       cmv_vi 1274           11.18        9.87      -1.31  -11.7 no — trained on this dataset
vlsp2020_100h  276           13.16        9.57      -3.59  -27.3 no — trained on this dataset


,dataset,n,baseline WER %,LoRA WER %,abs delta,rel %,zero-shot?
0,vivos,760,7.19,6.03,-1.15,-16.0,no — trained on this dataset
1,cmv_vi,1274,11.18,9.87,-1.31,-11.7,no — trained on this dataset
2,vlsp2020_100h,276,13.16,9.57,-3.59,-27.3,no — trained on this dataset
